# Nirasa 7B — Thai Language Model Training

Continued pretraining of **Qwen2.5-7B** on Thai corpora using LoRA on Colab A100.

**Requirements:** Colab Pro with A100 GPU runtime.

## What this notebook does:
1. Clone repo & install dependencies
2. Upload pre-processed training data (407M tokens)
3. Train LoRA (r=64) for 5000 steps
4. Save checkpoints to Google Drive
5. Test generation

## Step 0: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1: Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/worachetdee/nirasa.git /content/nirasa
!cd /content/nirasa && pip install torch transformers peft datasets accelerate -q

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Upload Training Data

Upload the pre-processed binary files from your local machine:
- `data/processed/th_all_qwen.bin` (1.6 GB)
- `data/processed/th_all_qwen.idx` (4.2 MB)

**Option A:** Upload via Colab file browser (drag & drop)

**Option B:** Upload to Google Drive first, then copy:

In [ ]:
import os
os.makedirs('/content/nirasa/data/processed', exist_ok=True)

# Option B: Copy from Google Drive (if you uploaded there first)
# Adjust the source path to match where you uploaded the files
DRIVE_DATA = '/content/drive/MyDrive/nirasa_data'

if os.path.exists(f'{DRIVE_DATA}/th_all_qwen.bin'):
    !cp {DRIVE_DATA}/th_all_qwen.bin /content/nirasa/data/processed/
    !cp {DRIVE_DATA}/th_all_qwen.idx /content/nirasa/data/processed/
    print('Copied from Google Drive')
else:
    print(f'Data not found at {DRIVE_DATA}/')
    print('Upload th_all_qwen.bin and th_all_qwen.idx to Google Drive,')
    print('or use the Colab file browser to upload directly.')

# Verify
for f in ['th_all_qwen.bin', 'th_all_qwen.idx']:
    path = f'/content/nirasa/data/processed/{f}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  {f}: {size_mb:.1f} MB')
    else:
        print(f'  {f}: MISSING')

## Step 3: Train

In [ ]:
import math
import struct
import time
from pathlib import Path

import numpy as np
import torch
from peft import LoraConfig, PeftModel, get_peft_model, TaskType
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================================
# Configuration — for 80GB A100
# ============================================================
MODEL_NAME = 'Qwen/Qwen2.5-7B'
DATA_BIN = '/content/nirasa/data/processed/th_all_qwen.bin'
DATA_IDX = '/content/nirasa/data/processed/th_all_qwen.idx'
OUTPUT_DIR = '/content/drive/MyDrive/nirasa_checkpoints/nirasa-7b-th-v3'

MAX_SEQ_LEN = 2048
BATCH_SIZE = 2
GRAD_ACCUM = 8       # effective batch = 16
LR = 2e-4
MAX_STEPS = 5000
SAVE_STEPS = 500
LOG_STEPS = 10
WARMUP = 100
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY = 0.01
SEED = 42

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

# Verify GPU has enough memory — STOP if not 80GB
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}, Memory: {gpu_mem:.0f} GB')
    if gpu_mem < 70:
        raise RuntimeError(
            f'Got {gpu_mem:.0f}GB GPU. This notebook requires 80GB A100. '
            f'Go to Runtime -> Disconnect and delete runtime, then reconnect until you get 80GB.'
        )
    print('80GB A100 confirmed.')

print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Tokens per step: {BATCH_SIZE * GRAD_ACCUM * MAX_SEQ_LEN:,}')
print(f'Total tokens (5000 steps): {MAX_STEPS * BATCH_SIZE * GRAD_ACCUM * MAX_SEQ_LEN:,}')

In [ ]:
# ============================================================
# Dataset
# ============================================================
class MemmapDataset(Dataset):
    def __init__(self, bin_path, idx_path, seq_len=2048):
        self.seq_len = seq_len
        with open(idx_path, 'rb') as f:
            header = f.read(12)
            num_docs, total_tokens, dtype_size = struct.unpack('<III', header)
        self.total_tokens = total_tokens
        self.data = np.memmap(bin_path, dtype=np.uint32, mode='r', shape=(total_tokens,))
        self.num_samples = max(0, (total_tokens - 1) // seq_len)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len + 1
        end = min(end, self.total_tokens)
        tokens = torch.from_numpy(self.data[start:end].astype(np.int64))
        input_ids = tokens[:-1]
        labels = tokens[1:]
        if len(input_ids) < self.seq_len:
            pad_len = self.seq_len - len(input_ids)
            input_ids = torch.cat([input_ids, torch.zeros(pad_len, dtype=torch.long)])
            labels = torch.cat([labels, torch.full((pad_len,), -100, dtype=torch.long)])
        return {'input_ids': input_ids, 'labels': labels}

dataset = MemmapDataset(DATA_BIN, DATA_IDX, seq_len=MAX_SEQ_LEN)
print(f'Dataset: {len(dataset):,} samples, {dataset.total_tokens:,} tokens')

In [ ]:
# ============================================================
# Checkpoint resume
# ============================================================
def find_latest_checkpoint(output_dir):
    output_path = Path(output_dir)
    if not output_path.exists():
        return None, 0
    checkpoints = []
    for d in output_path.iterdir():
        if d.is_dir() and d.name.startswith('step_'):
            try:
                step = int(d.name.split('_')[1])
                checkpoints.append((step, str(d)))
            except (ValueError, IndexError):
                continue
    if not checkpoints:
        return None, 0
    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1], checkpoints[-1][0]

resume_path, start_step = find_latest_checkpoint(OUTPUT_DIR)
if resume_path:
    print(f'Will resume from: {resume_path} (step {start_step})')
else:
    print('Starting fresh training')

In [ ]:
# ============================================================
# Load model + LoRA (bf16, full precision)
# ============================================================
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

if resume_path:
    print(f'Loading LoRA weights from {resume_path}')
    model = PeftModel.from_pretrained(model, resume_path, is_trainable=True)
else:
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
# ============================================================
# Training setup
# ============================================================
torch.manual_seed(SEED)
np.random.seed(SEED)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
)

def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps, min_lr_ratio=0.1):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_lr_ratio + (1.0 - min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP, MAX_STEPS)

# Restore optimizer/scheduler state if resuming
if resume_path:
    state_path = Path(resume_path) / 'training_state.pt'
    if state_path.exists():
        print(f'Restoring training state from {state_path}')
        training_state = torch.load(state_path, map_location='cpu', weights_only=True)
        optimizer.load_state_dict(training_state['optimizer'])
        scheduler.load_state_dict(training_state['scheduler'])
    else:
        print('No training_state.pt found, fast-forwarding scheduler')
        for _ in range(start_step):
            scheduler.step()

print('Training setup complete')

In [ ]:
# ============================================================
# Training loop
# ============================================================
model.train()
global_step = start_step
total_loss = 0.0
total_tokens_processed = 0
start_time = time.time()
log_start_time = time.time()
data_iter = iter(dataloader)

print(f'\nTraining: step {start_step} -> {MAX_STEPS}')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LR: {LR}, LoRA r={LORA_R}, alpha={LORA_ALPHA}')
print(f'  Seq len: {MAX_SEQ_LEN}, Save every: {SAVE_STEPS} steps')

while global_step < MAX_STEPS:
    optimizer.zero_grad()
    accum_loss = 0.0

    for _ in range(GRAD_ACCUM):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        input_ids = batch['input_ids'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item()
        total_tokens_processed += (labels != -100).sum().item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    scheduler.step()
    global_step += 1
    total_loss += accum_loss

    if global_step % LOG_STEPS == 0:
        elapsed = time.time() - log_start_time
        avg_loss = total_loss / LOG_STEPS
        current_lr = scheduler.get_last_lr()[0]
        tok_per_sec = total_tokens_processed / max(elapsed, 1e-6)
        total_elapsed = time.time() - start_time
        eta_sec = (MAX_STEPS - global_step) / (global_step - start_step) * total_elapsed if global_step > start_step else 0

        print(
            f'step {global_step:>6d}/{MAX_STEPS} | '
            f'loss {avg_loss:.4f} | '
            f'lr {current_lr:.2e} | '
            f'tok/s {tok_per_sec:.0f} | '
            f'elapsed {total_elapsed/3600:.1f}h | '
            f'ETA {eta_sec/3600:.1f}h'
        )

        total_loss = 0.0
        total_tokens_processed = 0
        log_start_time = time.time()

    if global_step % SAVE_STEPS == 0:
        ckpt_dir = Path(OUTPUT_DIR) / f'step_{global_step}'
        print(f'Saving checkpoint: {ckpt_dir}')
        model.save_pretrained(str(ckpt_dir))
        torch.save(
            {'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'step': global_step},
            str(ckpt_dir / 'training_state.pt'),
        )

# Save final
final_dir = Path(OUTPUT_DIR) / f'step_{global_step}'
print(f'Saving final: {final_dir}')
model.save_pretrained(str(final_dir))

total_time = time.time() - start_time
print(f'\nTraining complete! {global_step} steps in {total_time:.0f}s ({total_time/3600:.1f}h)')

## Step 4: Test Generation

In [ ]:
# Load tokenizer and test generation
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompts = [
    'ประเทศไทยมี',
    'กรุงเทพมหานครเป็น',
    'อาหารไทยที่มีชื่อเสียงที่สุดคือ',
    'ภาษาไทยเป็นภาษาที่',
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f'\n{"="*60}')
    print(f'Prompt: {prompt}')
    print(f'Output: {text}')

## Step 5: Compare with Base Model

In [ ]:
# Compare with base model
# First free memory from trained model
trained_model_path = str(Path(OUTPUT_DIR) / f'step_{global_step}')
del model
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
prompt = 'กรุงเทพมหานครเป็น'
inputs = tokenizer(prompt, return_tensors='pt')

# Generate with base model
print('=== Base Qwen2.5-7B (before training) ===')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
base_model.eval()
with torch.no_grad():
    output = base_model.generate(
        **inputs.to(base_model.device), max_new_tokens=200,
        temperature=0.7, top_p=0.9, repetition_penalty=1.2, do_sample=True,
    )
print(tokenizer.decode(output[0], skip_special_tokens=True))

# Generate with Nirasa (reload from checkpoint)
print(f'\n=== Nirasa 7B (after training) ===')
nirasa = PeftModel.from_pretrained(base_model, trained_model_path)
nirasa.eval()
with torch.no_grad():
    output = nirasa.generate(
        **inputs.to(nirasa.device), max_new_tokens=200,
        temperature=0.7, top_p=0.9, repetition_penalty=1.2, do_sample=True,
    )
print(tokenizer.decode(output[0], skip_special_tokens=True))

del base_model, nirasa
torch.cuda.empty_cache()